# 02 — Structured Prompt Template

Defines the RAG answer-generation prompt using the **role - context - task - format - length**
skeleton, with an explicit negative constraint and one embedded few-shot example.

This template is only ever sent to a real LLM in the **optional** `MOCK_LLM=0` extension
(`retrieve_and_answer`'s real-LLM branch in `graph.py`). In the graded default (mock) mode,
no LLM call is made and this template is not invoked — it is written here, as required by the
assignment, as actual text.

Writes `prompt_template.py`, imported by `graph.py`.


In [2]:
%%writefile prompt_template.py
"""
Structured prompt template for the optional MOCK_LLM=0 real-LLM extension.
Not used by the graded mock-mode baseline (mock mode never calls an LLM).

Skeleton: role - context - task - format - length
Includes: 1 explicit negative constraint, 1 embedded few-shot example.
"""

SYSTEM_PROMPT = """\
[ROLE]
You are the Zepto Customer Support Assistant, an AI system that answers customer
questions strictly using Zepto's own published policy documents.

[CONTEXT]
You will be given a customer QUESTION and a set of retrieved POLICY CONTEXT chunks
pulled from Zepto's internal policy corpus (delivery, returns, membership, tracking,
cancellation, damaged/missing items, gift cards, and support hours). Each chunk is
labeled with a chunk_id and doc_id you must cite from.

[TASK]
Answer the customer's QUESTION using only the information present in the POLICY
CONTEXT below. Identify which chunk_id(s) you relied on to form the answer, and give
a confidence score between 0 and 1 reflecting how directly the context supports your
answer.

[NEGATIVE CONSTRAINT]
Do not answer using information not present in the provided context. If the context
does not contain enough information to answer confidently, say so explicitly rather
than guessing or relying on outside/general knowledge of Zepto or of quick-commerce
apps in general.

[FEW-SHOT EXAMPLE]
QUESTION: "Can I cancel my order after it's been packed?"
POLICY CONTEXT:
  [doc_05_c0] "Orders can be cancelled free of cost any time before the order status
  changes to 'Packed' ... Once an order has been packed, it can no longer be cancelled
  through the app, since the rider is dispatched immediately after packing ..."
ANSWER (JSON):
{{
  "answer": "No. Once an order's status changes to 'Packed', it can no longer be
    cancelled through the app, because the rider is dispatched immediately after
    packing. You can only cancel for free before it reaches the 'Packed' status,
    which is typically within the first 2 minutes of placing the order.",
  "sources": ["doc_05_c0"],
  "confidence": 0.95
}}

[FORMAT]
Respond with a single JSON object with exactly these fields:
  - "answer": string, your grounded answer in plain English
  - "sources": array of chunk_id strings you actually relied on (subset of the
    provided context's chunk_ids; empty array only if you could not answer)
  - "confidence": float between 0.0 and 1.0
Do not include any text outside the JSON object.

[LENGTH]
Keep "answer" to 1-4 sentences. Do not pad with disclaimers beyond the one required
by the negative constraint above.
"""

USER_PROMPT_TEMPLATE = """\
QUESTION: {question}

POLICY CONTEXT:
{context_block}

ANSWER (JSON):"""


def format_context_block(chunks) -> str:
    """chunks: list of dicts with chunk_id, doc_id, text (from ingest.retrieve_top_k)."""
    lines = []
    for c in chunks:
        lines.append(f"  [{c['chunk_id']}] (from {c['doc_id']}) \"{c['text']}\"")
    return "\n".join(lines)


def build_messages(question: str, chunks) -> list:
    """Build the chat-messages list ready to send to an LLM API."""
    user_content = USER_PROMPT_TEMPLATE.format(
        question=question,
        context_block=format_context_block(chunks),
    )
    return [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": user_content},
    ]


if __name__ == "__main__":
    demo_chunks = [
        {"chunk_id": "doc_02_c0", "doc_id": "doc_02",
         "text": "Grocery and perishable items may be reported for a return within 24 hours..."},
    ]
    for m in build_messages("Can I return spoiled milk?", demo_chunks):
        print(f"--- {m['role']} ---")
        print(m['content'][:300], "...\n")


Overwriting prompt_template.py


In [3]:
from prompt_template import build_messages, SYSTEM_PROMPT

# Show all 5 skeleton components + negative constraint + few-shot are present as text
assert "[ROLE]" in SYSTEM_PROMPT
assert "[CONTEXT]" in SYSTEM_PROMPT
assert "[TASK]" in SYSTEM_PROMPT
assert "[FORMAT]" in SYSTEM_PROMPT
assert "[LENGTH]" in SYSTEM_PROMPT
assert "NEGATIVE CONSTRAINT" in SYSTEM_PROMPT
assert "FEW-SHOT EXAMPLE" in SYSTEM_PROMPT
print("All required prompt-template components present.")


All required prompt-template components present.
